# ShopSense 03: Hybrid search (keywords + meaning)

**Goal:** given a question, find the review passages that best answer it. This is the "R" in RAG (retrieval-augmented generation): the answer model will only see what this step finds.

**Why two kinds of search?**
- **BM25 (keywords):** great when the exact words matter: a brand, a model number, "paraben", "#2 blade". Weak when people use different words for the same idea.
- **Embeddings (meaning):** great for "makes my hair feel soft" matching "silky smooth". Weak on exact names and numbers.
- **Hybrid:** run both, then merge the two ranked lists with **Reciprocal Rank Fusion (RRF)**. A passage ranked high by either method rises to the top.

**Reads:** `data/processed/passages.parquet` (from notebook 02).
**Writes:** embeddings cache in `data/cache/`, a LanceDB table in `data/warehouse/lancedb/`, and small result files in `reports/search/`.

**Install once** (venv active): `pip install sentence-transformers lancedb rank-bm25`

## 1. Setup

**What this does:** imports libraries, finds the project root and sets the search settings in one place.

In [1]:
import json
import re
import time
from pathlib import Path

import lancedb
import numpy as np
import pandas as pd
import pyarrow as pa
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
CACHE = ROOT / "data" / "cache"
LANCE_DIR = ROOT / "data" / "warehouse" / "lancedb"
OUT = ROOT / "reports" / "search"
for folder in [CACHE, LANCE_DIR, OUT]:
    folder.mkdir(parents=True, exist_ok=True)

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # small and fast on a laptop CPU (384 numbers per text)
BATCH_SIZE = 64
CANDIDATES = 50      # how many results each method returns before merging
RRF_K = 60           # standard RRF constant: a bigger number makes ranks matter less
SHOW_TOP = 5         # how many merged results to print

d:\Resume\Data Scientist\Resume_Projects\ShopSense_Product Review Intelligence Copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the passages

**What this does:** loads the passages and resets the row numbers (row number = position in the BM25 and embedding arrays, so they must line up).

In [2]:
passages = pd.read_parquet(PROCESSED / "passages.parquet").reset_index(drop=True)
products = pd.read_parquet(PROCESSED / "products_clean.parquet")
print("passages:", len(passages))
print("products:", len(products), "| answerable:", int(products["answerable"].sum()))
passages[["passage_id", "parent_asin", "rating", "text"]].head(3)

passages: 50524
products: 24493 | answerable: 548


,passage_id,parent_asin,rating,text
0,r000000-0,B07MLZX6G5,5.0,"You have to have the know how to work THIS Wig. It looks Wild and almost untrainable but if you have ANYKIND of Styling talent, it’s fix..."
1,r000001-0,B072DV96BK,5.0,I love using this blush. So happy to find it on Amazon. Fast delivery.
2,r000002-0,B01IDQJF1W,4.0,Staring with the basics.....the packaging was just as pictured above. Very colorful as well as sturdy. The material was made out of thic...


## 3. Embeddings (search by meaning)

**What this does:** loads the embedding model. The first run downloads about 90 MB.

In [3]:
embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 256
DIM = embedder.get_sentence_embedding_dimension()
print("model:", EMBED_MODEL, "| numbers per text:", DIM)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 399.18it/s]


model: sentence-transformers/all-MiniLM-L6-v2 | numbers per text: 384


C:\Users\Dell\AppData\Local\Temp\ipykernel_2404\3368293892.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()


**What this does:** times 500 passages so you know how long the full run will take (a few minutes on a laptop CPU).

In [4]:
sample_texts = passages["search_text"].head(500).tolist()
start = time.perf_counter()
_ = embedder.encode(sample_texts, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=False)
seconds = time.perf_counter() - start
print(f"500 passages took {seconds:.1f} s -> all {len(passages)} would take about {seconds * len(passages) / 500 / 60:.1f} minutes")

500 passages took 23.2 s -> all 50524 would take about 39.1 minutes


**What this does:** embeds every passage (or loads the saved file if it was done before, so you only wait once). Vectors are normalised, so a dot product equals cosine similarity.

In [5]:
EMB_FILE = CACHE / "embeddings_minilm.npy"
IDS_FILE = CACHE / "embeddings_minilm_ids.json"

reuse = False
if EMB_FILE.exists() and IDS_FILE.exists():
    saved_ids = json.loads(IDS_FILE.read_text())
    if saved_ids == passages["passage_id"].tolist():
        reuse = True

if reuse:
    embeddings = np.load(EMB_FILE)
    print("loaded saved embeddings:", embeddings.shape)
else:
    embeddings = embedder.encode(passages["search_text"].tolist(), batch_size=BATCH_SIZE,
                                 normalize_embeddings=True, show_progress_bar=True)
    embeddings = np.asarray(embeddings, dtype="float32")
    np.save(EMB_FILE, embeddings)
    IDS_FILE.write_text(json.dumps(passages["passage_id"].tolist()))
    print("saved embeddings:", embeddings.shape)

Batches: 100%|██████████| 790/790 [36:10<00:00,  2.75s/it]  


saved embeddings: (50524, 384)


## 4. LanceDB (where the vectors live)

**What this does:** stores each passage's id, product, rating and vector in a LanceDB table on disk. With about 50,000 vectors, an exact (brute-force) search is already fast, so we do not add an approximate index. The vectors are written in Arrow format to avoid huge Python lists.

In [6]:
meta = passages[["passage_id", "parent_asin", "rating", "is_short"]].copy()
vector_column = pa.FixedSizeListArray.from_arrays(pa.array(embeddings.reshape(-1)), DIM)
data = pa.Table.from_pandas(meta, preserve_index=False).append_column("vector", vector_column)

db = lancedb.connect(str(LANCE_DIR))
table = db.create_table("passages", data=data, mode="overwrite")
print("rows in LanceDB table:", table.count_rows())

rows in LanceDB table: 50524


## 5. BM25 (search by keywords)

**What this does:** splits every passage into lowercase words and builds the BM25 index. Also builds a lookup from product to its row numbers, so we can search inside one product.

In [7]:
token_pattern = re.compile(r"[a-z0-9']+")
corpus_tokens = [token_pattern.findall(text.lower()) for text in passages["search_text"]]
bm25 = BM25Okapi(corpus_tokens)
rows_of_product = passages.groupby("parent_asin").indices
print("documents indexed:", len(corpus_tokens))
print("products in lookup:", len(rows_of_product))

documents indexed: 50524
products in lookup: 24493


## 6. Search with all three methods

**What this does:** for each question it runs **vector search**, **BM25 search**, and **hybrid (RRF)**, and prints the top 3 of each side by side with timings.

Each entry is `(question, product_id or None)`. `None` searches all reviews; a product id searches only that product's reviews. Change the list and rerun to try your own questions.

In [8]:
answerable = products[products["answerable"]].sort_values("clean_reviews", ascending=False)
BIG_PRODUCT = answerable.iloc[0]["parent_asin"]
print("product used for the scoped example:", answerable.iloc[0]["title"][:80], "| reviews:", answerable.iloc[0]["clean_reviews"])

product used for the scoped example: Salux Nylon Japanese Beauty Skin Bath Wash Cloth/towel (3) Blue Yellow and Pink | reviews: 178


In [9]:
QUERIES = [
    ("does it irritate sensitive skin", None),
    ("makes my hair feel soft and silky", None),
    ("eyebrow razor", None),
    ("is it worth the money", BIG_PRODUCT),
]

In [10]:
text_of = dict(zip(passages["passage_id"], passages["text"]))
stars_of = dict(zip(passages["passage_id"], passages["rating"]))
log_rows = []

for question, asin in QUERIES:
    print("=" * 100)
    print("QUESTION:", question, "| product filter:", asin)

    # ---- vector search
    t0 = time.perf_counter()
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    search = table.search(question_vector).metric("cosine").limit(CANDIDATES)
    if asin is not None:
        search = search.where(f"parent_asin = '{asin}'", prefilter=True)
    vector_ids = search.to_pandas()["passage_id"].tolist()
    t_vector = time.perf_counter() - t0

    # ---- bm25 search
    t0 = time.perf_counter()
    scores = bm25.get_scores(token_pattern.findall(question.lower()))
    if asin is None:
        order = np.argsort(-scores)[:CANDIDATES]
    else:
        rows = rows_of_product[asin]
        order = rows[np.argsort(-scores[rows])][:CANDIDATES]
    bm25_ids = [passages.at[i, "passage_id"] for i in order if scores[i] > 0]
    t_bm25 = time.perf_counter() - t0

    # ---- reciprocal rank fusion: each list gives 1 / (RRF_K + rank) points
    fused = {}
    for rank, passage_id in enumerate(vector_ids):
        fused[passage_id] = fused.get(passage_id, 0) + 1 / (RRF_K + rank + 1)
    for rank, passage_id in enumerate(bm25_ids):
        fused[passage_id] = fused.get(passage_id, 0) + 1 / (RRF_K + rank + 1)
    hybrid_ids = sorted(fused, key=fused.get, reverse=True)

    print(f"time: vector {1000 * t_vector:.0f} ms | bm25 {1000 * t_bm25:.0f} ms")
    for name, ids in [("VECTOR", vector_ids), ("BM25", bm25_ids), ("HYBRID", hybrid_ids)]:
        print(f"--- {name} top 3")
        for passage_id in ids[:3]:
            print(f"   [{passage_id}] {stars_of[passage_id]:.0f}* {text_of[passage_id][:110]}")

    for rank, passage_id in enumerate(hybrid_ids[:SHOW_TOP]):
        found_by = ("both" if passage_id in vector_ids and passage_id in bm25_ids
                    else "vector" if passage_id in vector_ids else "bm25")
        log_rows.append({"question": question, "product": asin, "rank": rank + 1, "passage_id": passage_id,
                         "found_by": found_by, "stars": stars_of[passage_id], "text": text_of[passage_id]})

results = pd.DataFrame(log_rows)
results.to_csv(OUT / "03_example_results.csv", index=False)
print("\nsaved", len(results), "rows to reports/search/03_example_results.csv")

QUESTION: does it irritate sensitive skin | product filter: None
time: vector 1775 ms | bm25 196 ms
--- VECTOR top 3
   [r046312-0] 1* Harsh on skin and doesn’t seam to really cleanse anything, leaves more of a film on the skin than anything els
   [r037233-0] 3* Packaging and application are great. Very sturdy cute and easy to use. Unfortunately the product irritated my 
   [r045961-0] 5* When skin is stressed, every little thing bothers it. Too much heat, too much cold, too much wind, too much dr
--- BM25 top 3
   [r022162-0] 4* I got sensitive skin and I am very careful with skin care. This doesn't irritate my skin, it doesn't clog pore
   [r038853-0] 4* Good face wash. It removes the oil without drying out your skin. I have fair sensitive combination skin that g
   [r010237-0] 5* This citrus scent is so nice. It's light and not too strong and does not irritate my sensitive skin. It cleans
--- HYBRID top 3
   [r034781-0] 5* I got this from IPSY as part of my makeup subscription. It 

**What this does:** shows, for the hybrid top results, how many were found by only one method. This is the evidence for why hybrid is worth having.

In [11]:
print(results["found_by"].value_counts())
results.groupby("question")["found_by"].value_counts().unstack(fill_value=0)

found_by
both      12
vector     4
bm25       4
Name: count, dtype: int64


found_by,bm25,both,vector
question,,,
does it irritate sensitive skin,2,1,2
eyebrow razor,0,5,0
is it worth the money,0,5,0
makes my hair feel soft and silky,2,1,2


## 7. Save the settings

**What this does:** records exactly how the index was built, so the next notebooks and the README can quote it.

In [12]:
config = {
    "embedding_model": EMBED_MODEL, "embedding_dim": int(DIM), "max_seq_length": 256,
    "passages_indexed": len(passages), "lancedb_table": "passages", "metric": "cosine",
    "bm25": "rank_bm25.BM25Okapi over product title + review title + text",
    "fusion": "reciprocal rank fusion", "rrf_k": RRF_K, "candidates_per_method": CANDIDATES,
}
(OUT / "03_search_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))

{
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "max_seq_length": 256,
  "passages_indexed": 50524,
  "lancedb_table": "passages",
  "metric": "cosine",
  "bm25": "rank_bm25.BM25Okapi over product title + review title + text",
  "fusion": "reciprocal rank fusion",
  "rrf_k": 60,
  "candidates_per_method": 50
}


## How to explain this in an interview

- "Retrieval quality caps answer quality, so I built search first and measured it before adding any language model."
- "I used **hybrid search**: BM25 for exact words like brands and model numbers, and sentence embeddings for meaning. I merged them with reciprocal rank fusion, which needs no tuning of score scales, because it uses only ranks."
- "I stored vectors in LanceDB on disk with a product filter, so a question about one product only searches that product's reviews. With 50,000 passages an exact search is fast enough, so I skipped an approximate index."
- "I chose a small embedding model (384 dimensions) that runs on a laptop CPU, and I cached the embeddings so the slow step runs once."
- "This notebook only shows examples. The next notebook measures retrieval with a labelled question set, so I can say how often the right review is in the top 5 for each method."